# Cable Network Analysis & Prioritization

## Overview
Analysis of 21,800 cable segments to identify high-priority maintenance targets using time series analysis.

## Business Context
**Problem:** Develop a preventive maintenance plan that allows prioritizing and categorizing the different situations of the cable segments, based on relating different tables with information and time-analysis.

**Impact:** This categorization of medium-voltage cable segments results in a comprehensive document with accurate information, which will then be analyzed by different geographical areas to conduct measurement tests according to budget. This has increased investment efficiency, allowing more work to be done at the same cost.

This notebook reads input files from `../data/input/` and generates output in `../data/output/`

**Input data sources:**
- `4-Etapas.csv` - Network staging data
- `Base Anomalias-Averias MT.xlsx` - Historical failures and anomalies
- `Dicc_INV.xlsx` - Segment (MM) dictionary to link different data tables
- `Elementos_PYO.xlsx` - Planning and project elements
- `Inventario_MT.parquet` - Complete network inventory

**Output:**
- `Plan26.xlsx` - Prioritized maintenance plan with 284 high-priority segments

## Process
1. Load and consolidate data from multiple sources
2. Perform time series analysis on failure patterns
3. Calculate risk scores based on multiple factors
4. Categorize and prioritize segments
5. Generate formatted Excel output

## Glossary

- **MM (Módulo de Mantenimiento)**: Maintenance segment identifier (21,800 total segments in network)
- **TPLMA**: Technical location ID for each network segment
- **UT (Unidad Técnica)**: Technical unit in the inventory system
- **Empalme**: Cable joint/splice point
- **API**: Paper-Insulated cable type
- **SECO (XLPE)**: Dry-insulated cable type

## Case Categories Explained

Network segments are evaluated and assigned to one of 7 categories:

1. **Work in Progress** - Segments already repaired after 01/01/2023 or currently under maintenance
2. **Action Required** - Latest diagnostic test failed (needs engineering evaluation)
3. **Test Recommended** - Had failure AFTER passing a diagnostic test (deterioration suspected)
4. **Test Recommended** - 2+ failures recorded (needs diagnostic testing)
5. **Review Required** - 2+ failures BUT recent test was successful (monitor closely)
6. **Feeder Review** - Feeder has ≥3 failures but no specific MM candidates identified
7. **Not a Candidate** - Does not meet priority criteria

*Note: Categories 1-6 are considered high-priority. Category 7 segments are excluded from the maintenance plan.*



# Read and filter databases

In [ ]:
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
import re
from datetime import datetime
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo

pd.reset_option('display.max_rows', None)
pd.reset_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

# Input and output directory
directorio = '../data/input'
output_dir = '../data/output'

## Read inventory and build MM base

In [ ]:
#KEEP ONLY TIPO_OBJ_TEC THAT BELONGS TO AN MM

INV_MM = pd.read_parquet(directorio + "/Inventario_MT.parquet")

INV_MM['F_ALTA_SAP']=pd.to_datetime(INV_MM['F_ALTA_SAP'],dayfirst=True)
INV_MM['F_BAJA_SAP']=pd.to_datetime(INV_MM['F_BAJA_SAP'],dayfirst=True)
INV_MM['PTA_SERVICIO']=pd.to_datetime(INV_MM['PTA_SERVICIO'],format='%d/%m/%Y', dayfirst=True)

INV_MM1=INV_MM.copy()

objetos=['EMPALME MT','CABLE MT','LINEA MT','PIQUETE MT']
INV_MM1 = INV_MM[INV_MM['TIPO_OBJ_TEC'].isin(objetos)].copy()
INV_MM1['LONGITUD'] = INV_MM1['LONGITUD'].fillna(0)
INV_MM1['LONGITUD'] = INV_MM1['LONGITUD'].astype(int)

INV_zonas = INV_MM1.copy()
INV_zonas = (INV_zonas.groupby(['TPLMA', 'AREA'])['LONGITUD']
                        .sum()
                        .reset_index()
                        .loc[lambda df: df.groupby('TPLMA')['LONGITUD'].idxmax()])

INV_zonas = INV_zonas.drop(columns='LONGITUD')


# Grouping and sum 'LONGITUD'
grouped_total = INV_MM1.groupby(['TPLMA', 'ME_NOMBRE', 'ME_ORDEN', 'EMPLAZAMIENTO', 'MPROT'])['LONGITUD'].sum().reset_index()

# Pivot to obtain the length by TIPO_OBJ_TEC
pivot_longitud_tipo = INV_MM1.pivot_table(index=['TPLMA', 'ME_NOMBRE', 'ME_ORDEN', 'EMPLAZAMIENTO', 'MPROT'], 
                                     columns='TIPO_OBJ_TEC', 
                                     values='LONGITUD', 
                                     aggfunc='sum',
                                     fill_value=0).reset_index()

pivot_detalle_tipo = INV_MM1.pivot_table(index=['TPLMA', 'ME_NOMBRE', 'ME_ORDEN', 'EMPLAZAMIENTO', 'MPROT'], 
                                     columns='detalle', 
                                     values='LONGITUD', 
                                     aggfunc='sum',
                                     fill_value=0).reset_index()

final_df = pd.merge(grouped_total, pivot_longitud_tipo, on=['TPLMA', 'ME_NOMBRE', 'ME_ORDEN', 'EMPLAZAMIENTO', 'MPROT'])
final_df = pd.merge(final_df, pivot_detalle_tipo, on=['TPLMA', 'ME_NOMBRE', 'ME_ORDEN', 'EMPLAZAMIENTO', 'MPROT'])

final_df = final_df.drop(columns=['API-API', 'SECO-SECO', 'Sin secuencia', 'TRANSICION', 'EMPALME MT'])

final_df = pd.merge(final_df, INV_zonas, how='left', on='TPLMA')

INV_tipo_emp = INV_MM1[INV_MM1['TIPO_OBJ_TEC'] == 'EMPALME MT'].pivot_table(
    index='TPLMA', 
    columns='detalle',
    values='TIPO_OBJ_TEC',
    aggfunc='count',
    fill_value=0
).reset_index()

INV_count_emp = INV_MM1[INV_MM1['TIPO_OBJ_TEC'] == 'EMPALME MT'].pivot_table(
    index='TPLMA', 
    columns='TIPO_OBJ_TEC',
    values='detalle',
    aggfunc='count',
    fill_value=0
).reset_index()

final_df = pd.merge(final_df, INV_tipo_emp, how='left', on='TPLMA')
final_df = pd.merge(final_df, INV_count_emp, how='left', on='TPLMA')
final_df['API-API'] = final_df['API-API'].fillna(0).astype(int)
final_df['SECO-SECO'] = final_df['SECO-SECO'].fillna(0).astype(int)
final_df['Sin secuencia'] = final_df['Sin secuencia'].fillna(0).astype(int)
final_df['TRANSICION'] = final_df['TRANSICION'].fillna(0).astype(int)
final_df['EMPALME MT'] = final_df['EMPALME MT'].fillna(0).astype(int)
final_df = final_df.drop_duplicates(subset=['TPLMA'])

In [25]:
final_df.head(10)

,TPLMA,ME_NOMBRE,ME_ORDEN,EMPLAZAMIENTO,MPROT,LONGITUD,CABLE MT,LINEA MT,PIQUETE MT,API,...,SECO,TRIANGULAR,VERTICAL DESN,VERTICAL PROT,AREA,API-API,SECO-SECO,Sin secuencia,TRANSICION,EMPALME MT
0,MM100003356,MEC-77807-1|79438-3|,11,11114A,MP_11114A,104,104,0,0,0,...,104,0,0,0,1CA,0,0,0,0,0
1,MM100028113,MEL-B15-0601|S581JU|,4,45116A,MP_K15-0603,54,0,54,0,0,...,0,0,54,0,2LM,0,0,0,0,0
2,MM100031510,MEL-K27-1|K8.440MO|,6,6714A,MP_K27-1,179,0,179,0,0,...,0,0,0,0,2MO,0,0,0,0,0
3,MM100031533,MEC-1335-1|1766-3|,2,5015A,MP_5015A,1013,1013,0,0,0,...,1013,0,0,0,1OL,0,5,0,0,5
4,MM100032036,MEC-1236-5|S01-1152|,16,5537A,MP_5537A,170,170,0,0,0,...,170,0,0,0,1OL,0,0,0,0,0
5,MM100032346,MEL-S34-1462|Z34-1461|,5,25313A,MP_Z34-1461,10,4,6,0,0,...,4,0,0,0,3PI,0,0,0,0,0
6,MM100032347,MEC-35042-1|S34-1462|,6,25313A,MP_Z34-1461,71,71,0,0,0,...,71,0,0,0,3PI,0,0,0,0,0
7,MM100033951,MEL-B34-1514|S34-1466|,18,95353A,MP_95353A,4,0,4,0,0,...,0,0,0,0,3PI,0,0,0,0,0
8,MM100034055,MEL-13224-9|K13-1|S13-0239|,7,13212A,MP_K13-1,427,0,427,0,0,...,0,0,0,0,2ME,0,0,0,0,0
9,MM100034108,MEC-S07-2|S7.67MO|,7,6213A,MP_6213A,46,46,0,0,42,...,4,0,0,0,2ME,0,0,0,1,1


## Segment Dictionary

In [26]:
#Leo el diccionario para cruzar ponerle el MM a los vínculos
archivo='Dicc_INV.xlsx'
nombre_tabla = 'Dict_tramos_tabla'

# Cargar el archivo Excel con openpyxl
dicc_MM = pd.read_excel(directorio+'/'+archivo)

dicc_MM['Key']=dicc_MM['Key'].astype(str)

## Failures Database

In [27]:
averias = pd.read_excel(directorio + '/Base Anomalías-Averías MT.xlsx')
averias = averias[(averias['Form.Tipo de Instalación']=='Cable Subterráneo MT') & (averias['Elemento Averiado'].str.contains('<=>'))]
columns=['Anomalía','Fecha detección','Elemento Averiado','Tipo de Instalación','Tipo de Elemento', 'Form.Tipo de Elemento',
       'Form.Subtipo Elemento', 'Form.Marca', 'Form.Motivo de avería']
averias = averias[columns]
averias = averias[averias['Fecha detección'] >= pd.to_datetime('01/01/2024',format = '%d/%m/%Y')]

In [ ]:
# Merging failures with dictionary
averias = pd.merge(averias,dicc_MM[['Key','MM']],left_on='Elemento Averiado',right_on='Key',how='left')
averias = averias.drop_duplicates(subset=['Anomalía'])

## Stages

In [29]:
etapas = pd.read_csv(directorio+'/4-Etapas.csv')

## Projects and works in segments

In [ ]:
saneamientos = pd.read_excel(directorio + '/Elementos_PYO.xlsx',header=1)

saneamientos1 = saneamientos[(saneamientos['Categoría']=='Saneamiento') & (saneamientos['Tramo'].notna()) & (saneamientos['Tipo']=='Media Tensión')]
columns = ['Nombre iniciativa','Tramo', 'ID de iniciativa', 'Ini_ord', 'Tipo_1', 'Status',
         'Avance Físico %', 'Fe.inicio prevista','Fin real','Inic.real',  'Área Ejecutante']
saneamientos1 = saneamientos1[columns]
saneamientos1['Avance Físico %']=pd.to_numeric(saneamientos1['Avance Físico %'],errors = 'coerce')
saneamientos1['Avance Físico %'] = saneamientos1['Avance Físico %'].fillna(0)
saneamientos1['Tramo'] = saneamientos1['Tramo'].str.split('_')
saneamientos1 = saneamientos1.explode('Tramo').reset_index(drop=True)
saneamientos1 = saneamientos1.drop_duplicates(subset=['Tramo','ID de iniciativa'])
saneamientos1 = saneamientos1[saneamientos1['Fe.inicio prevista'] >= pd.to_datetime('01/01/2023',format = '%d/%m/%Y')]

saneamientos1['Key']=saneamientos1['ID de iniciativa']+';'+saneamientos1['Tramo']       #Build key for merging with dictionary
saneamientos1 = pd.merge(saneamientos1,dicc_MM[['Key','MM']],on='Key',how='left')
saneamientos1 = saneamientos1.drop_duplicates(subset=['Tramo','ID de iniciativa'])

# Grouping variables from different databases in plan dataframe

In [ ]:
# MM dataframe with geographical information
plan25 = final_df[['TPLMA','AREA','ME_NOMBRE','ME_ORDEN','EMPLAZAMIENTO',
                   'MPROT','LONGITUD','CABLE MT','EMPALME MT','LINEA MT',
                   'API','SECO', 'API-API', 'SECO-SECO', 'Sin secuencia',
                   'TRANSICION']]

# Adding MMs failures to dataframe
averias_plan = averias.copy()
averias_plan = averias_plan[averias_plan['Fecha detección']>=pd.to_datetime('01/04/2025',format='%d/%m/%Y')]
averias_plan = averias_plan[averias_plan['Fecha detección']<=pd.to_datetime('10/03/2026',format='%d/%m/%Y')]
averias_plan = averias_plan.drop_duplicates(subset=['Anomalía'])
grouped_averias = (
    averias_plan.groupby('MM').agg(
        Cant_averias = ('Anomalía', 'count'),
        Cant_cable = ("Form.Tipo de Elemento", lambda x: sum(x.str.contains("Cable", case=False, na=False))),
        Cant_emp = ("Form.Tipo de Elemento", lambda x: sum(x.str.contains("Empalme", case=False, na=False))),
        Cant_term = ("Form.Tipo de Elemento", lambda x: sum(x.str.contains("Terminal", case=False, na=False)))
    )
)

plan25 = pd.merge(plan25, grouped_averias, how='left', left_on='TPLMA', right_on='MM').fillna(0)

# Adding feeder failures
averias_plan2 = pd.merge(averias_plan, final_df[['TPLMA','EMPLAZAMIENTO']], how='left', left_on='MM',right_on='TPLMA')
averias_plan2 = averias_plan2.drop_duplicates(subset=['Anomalía'])
grouped_av_emp = (
    averias_plan2.groupby('EMPLAZAMIENTO').agg(
        Cant_averias_alim = ('Anomalía', 'count')
    )
).reset_index()
plan25 = pd.merge(plan25, grouped_av_emp, how='left', on='EMPLAZAMIENTO').fillna(0)


# Adding last stage of VLF and TgDelta tests
etapas_plan = etapas.copy()
etapas_plan = etapas_plan[etapas_plan['Tabla']=="Medicion"]
etapas_plan = etapas_plan.loc[etapas_plan.groupby("MM")["Time"].idxmax()]
plan25 = pd.merge(plan25, etapas_plan[['MM','Time','Tipo','Tipo2','Tiene_averias']], how='left', left_on='TPLMA',right_on='MM').fillna(0)
plan25 = plan25.drop(columns = 'MM')

# Adding projects saneamientos
saneam_plan = saneamientos1.copy()
plan25 = pd.merge(plan25, saneam_plan[['MM','ID de iniciativa','Status','Inic.real']], how='left', left_on='TPLMA',right_on='MM').fillna(0)
plan25 = plan25.drop(columns = 'MM')


# Creo columnas con condiciones

## Risk Categorization Logic

Segments are classified into 7 priority categories based on:
1. **Existing maintenance work** (Category 1)
2. **Test results** - Failed diagnostic tests (Category 2)
3. **Failure patterns** - 2+ failures with different test outcomes (Categories 3, 4, 5)
4. **Feeder-level patterns** - Feeders with 3+ failures but no specific segment identified (Category 6)
5. **Not a candidate** - Segments that don't meet risk criteria (Category 7)

This logic comes from engineering best practices that prioritize segments with:
- Recent failures AND poor test results
- Multiple failures indicating deterioration
- High-failure feeders that need investigation


In [32]:
plan25['2Averías'] = plan25['Cant_averias'].apply(lambda x: 'Si' if x>1 else 'No')
plan25['Ensayo_mal'] = plan25['Tipo2'].apply(lambda x: 'Si' if x =="Cayó en prueba" or "Riesgo Alto" in str(x) else 'No' )
plan25['Ensayo_aver'] = plan25.apply(lambda row: 'Si' if row['Ensayo_mal'] == 'No' and row['Tiene_averias'] == True else 'No', axis=1)
plan25['Inic.real'] = pd.to_datetime(plan25['Inic.real'])
plan25['Sanem'] = np.where(
    (plan25['Inic.real'] >= pd.to_datetime('01/01/2025', format='%d/%m/%Y')) | 
    (~plan25['Status'].astype(str).isin(['59 - Finalizada', '60 - Cerrada', '0'])), 
    'Si', 
    'No'
)

def categoria_plan(row):
    if row['Sanem'] == 'Si':
        return '1- Saneamiento en curso'
    elif row['Ensayo_mal'] == 'Si':
        return '2- Evaluar acción a tomar - Con último ensayo mal'
    elif row['Ensayo_aver'] == 'Si':
        return '3- Ensayar - Con ensayo bien y avería posterior'
    elif row['2Averías'] == 'Si' and row['Ensayo_mal'] == 'No' and row['Tipo2'] != 0:
        return  '5- A revisar - Con 2 o más averías pero ensayo posterior exitoso'
    elif row['2Averías'] == 'Si':
        return '4- Ensayar - 2 o mas averías'
    else:
        return '7- No es candidato'

plan25['Categoría_plan'] = plan25.apply(categoria_plan, axis=1)

In [33]:
emplaz_no_plan = (plan25.groupby('EMPLAZAMIENTO')
             .agg(Cant_averias_alim=('Cant_averias_alim', 'mean'),
                  Cat_no_7=('Categoría_plan', lambda x: (~x.str.startswith('7')).sum()))
             .reset_index())

In [34]:
## Estado 6- A revisar - Alimentador con averías sin MM candidatos
emplaz_no_plan = (plan25.groupby('EMPLAZAMIENTO')
             .agg(Cant_averias_alim=('Cant_averias_alim', 'mean'),
                  Cat_no_7=('Categoría_plan', lambda x: (~x.str.startswith('7')).sum()))
             .reset_index())

emplaz_no_plan = emplaz_no_plan[(emplaz_no_plan['Cant_averias_alim'] >=3) & (emplaz_no_plan['Cat_no_7'] == 0)]

emplaz_no_plan['Categ'] = '6- A revisar - Alimentador con averías sin MM candidatos'
emplaz_no_plan = emplaz_no_plan[['EMPLAZAMIENTO','Categ']]
emplaz_no_plan = emplaz_no_plan.drop_duplicates(subset='EMPLAZAMIENTO')

plan25 = pd.merge(plan25, emplaz_no_plan, how='left', on='EMPLAZAMIENTO')

plan25['Categoría_plan'] = np.where(plan25['Categ'].notna(), 
                                       plan25['Categ'], 
                                       plan25['Categoría_plan'])

plan25 = plan25.drop(columns = 'Categ')
plan25 = plan25.drop_duplicates(subset=['TPLMA'])

In [35]:
plan25[plan25['EMPLAZAMIENTO'] == '15311A'][['EMPLAZAMIENTO', 'Cant_averias_alim', 'Categoría_plan']].groupby('EMPLAZAMIENTO').agg(Cant_averias_alim=('Cant_averias_alim', 'mean'),
                  Cat_no_7=('Categoría_plan', lambda x: (~x.str.startswith('7')).sum())).reset_index()

,EMPLAZAMIENTO,Cant_averias_alim,Cat_no_7
0,15311A,7.0,3


In [36]:
plan25

,TPLMA,AREA,ME_NOMBRE,ME_ORDEN,EMPLAZAMIENTO,MPROT,LONGITUD,CABLE MT,EMPALME MT,LINEA MT,...,Tipo2,Tiene_averias,ID de iniciativa,Status,Inic.real,2Averías,Ensayo_mal,Ensayo_aver,Sanem,Categoría_plan
0,MM100003356,1CA,MEC-77807-1|79438-3|,11,11114A,MP_11114A,104,104,0,0,...,Prueba exitosa,False,0,0,1970-01-01,No,No,No,No,7- No es candidato
1,MM100028113,2LM,MEL-B15-0601|S581JU|,4,45116A,MP_K15-0603,54,0,0,54,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
2,MM100031510,2MO,MEL-K27-1|K8.440MO|,6,6714A,MP_K27-1,179,0,0,179,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
3,MM100031533,1OL,MEC-1335-1|1766-3|,2,5015A,MP_5015A,1013,1013,5,0,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
4,MM100032036,1OL,MEC-1236-5|S01-1152|,16,5537A,MP_5537A,170,170,0,0,...,Riesgo Moderado,False,0,0,1970-01-01,No,No,No,No,7- No es candidato
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21878,MM99993617,2MO,MEL-28119-9|28191-9|8304-9|8515-9|K8.910MO|S28...,3,6423A,MP_6423A,1187,9,0,1178,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
21879,MM99993639,2ME,MEL-S13-0262|,53,13211A,MP_K13-50MO,17,15,0,2,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
21880,MM99998715,3PI,MEL-B34-1467|S34-1468|,22,25322A,MP_R34-396,3,0,0,3,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato
21881,MM99998716,3PI,MEL-34187-9|34225-9|34961-9|B34-1467|S34-0374|,23,25322A,MP_R34-396,1179,0,0,1179,...,0,0,0,0,1970-01-01,No,No,No,No,7- No es candidato


In [18]:
emplaz_no_plan[emplaz_no_plan['EMPLAZAMIENTO'] == '15311A']

,EMPLAZAMIENTO,Categ


# Exporto el plan

In [19]:
# Crear un archivo Excel y una hoja de trabajo
wb = Workbook()
ws = wb.active
ws.title = "Plan"

# Insertar el DataFrame en la hoja
for r_idx, row in enumerate(dataframe_to_rows(plan25, index=False, header=True), start=1):
    ws.append(row)

# Definir el rango de la tabla (incluyendo el encabezado)
num_rows, num_cols = plan25.shape
last_col_letter = get_column_letter(num_cols)  # Obtiene la letra de la última columna
table_range = f"A1:{last_col_letter}{num_rows + 1}"  # Define el rango correctamente

# Crear una tabla en la hoja de Excel
tabla = Table(displayName="Tabla1", ref=table_range)

# Estilo de la tabla
style = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False,
                       showLastColumn=False, showRowStripes=True, showColumnStripes=True)
tabla.tableStyleInfo = style

# Agregar la tabla a la hoja
ws.add_table(tabla)

# Ajustar automáticamente el ancho de las columnas
for col in ws.columns:
    max_length = max(len(str(cell.value)) if cell.value else 0 for cell in col) + 2
    ws.column_dimensions[col[0].column_letter].width = max_length

# Aplicar zoom al 80%
ws.sheet_view.zoomScale = 80

# Guardar el archivo Excel
wb.save(output_dir + '/Plan26.xlsx')

In [288]:
plan25['Categoría_plan'].value_counts()

Categoría_plan
7- No es candidato                                                  21428
1- Saneamiento en curso                                                88
2- Evaluar acción a tomar - Con último ensayo mal                      74
6- A revisar - Alimentador con averías sin MM candidatos               72
4- Ensayar - 2 o mas averías                                           68
3- Ensayar - Con ensayo bien y avería posterior                        65
5- A revisar - Con 2 o más averías pero ensayo posterior exitoso        5
Name: count, dtype: int64

### Network Segment Summary

The `final_df` dataframe contains one row per network segment (MM) with:
- **Geometry columns**: `LONGITUD` (total length), `CABLE MT`, `LINEA MT` (overhead line), `PIQUETE MT` (poles)
- **Cable type breakdown**: `API` (paper-insulated), `SECO` (dry-insulated), `COMPACTA`, etc.
- **Joint types**: `API-API`, `SECO-SECO`, `TRANSICION` (transition joints that mix cable types)
- **Geographic info**: `AREA`, `EMPLAZAMIENTO` (feeder ID), `MPROT` (protection module)

**Sample data shows:**
- Segments range from 4m to 1,013m in length
- Mix of underground cable (CABLE MT) and overhead lines (LINEA MT)
- Most segments have 0-5 joints (EMPALME MT)


## Results & Validation

### Summary Statistics:

Key Findings:
* 284 segments identified for priority maintenance (out of 21,800 total)
* This represents approximately X% of the network by length
* Category 2 (failed tests) and Category 4 (multiple failures) are the largest groups
* Geographic concentration in areas: [mention top 3 areas if applicable]
Next Steps:
The output file Plan26.xlsx is ready for engineering review and resource allocation.

In [ ]:
print("Category Distribution:")
print(plan25['Categoría_plan'].value_counts().sort_index())
print(f"\nTotal high-priority segments (Categories 1-6): {len(plan25[~plan25['Categoría_plan'].str.startswith('7')])}")
print(f"Total segment length under maintenance: {plan25[~plan25['Categoría_plan'].str.startswith('7')]['LONGITUD'].sum() / 1000:.1f} km")